# பாடம் 10 - உற்பத்தியில் AI முகவர்கள்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework (Python) பயன்படுத்தி AI முகவர்களுக்கான **உற்பத்தி மாதிரிகள்** பற்றி கற்றுக்கொள்வீர்கள். நாம் விவாதிப்பது:

- **Observability** — முகவர் தொடர்புகளில் நேரமிடுதல் மற்றும் பதிவு சேர்க்குதல்
- **Evaluation** — பதிலின் தரத்தை மதிப்பிட ஒரு மதிப்பீட்டர் முகவரை பயன்படுத்துதல்
- **Cost management** — டோக்கன் சிறந்த பயன்பாடு மற்றும் மாடல் தேர்விற்கான நடைமுறைகள்

தற்போதைய காட்சி என்பது பயனர்களுக்கு பயணத் திட்டங்களை அமைக்க உதவும் ஒரு **பயண முகவர்**, அதில் கண்காணிப்பு மற்றும் மதிப்பீடு மேல்நிலையாக சேர்க்கப்பட்டுள்ளது.


## அமைப்பு


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## உற்பத்தி தொடர்பான கருத்துக்கள்

AI முகவரிகளை ஆரம்ப மாதிரிகளிலிருந்து உற்பத்திக்கு கொண்டு செல்லும்போது மூன்று முக்கிய தூண்களுக்கு கவனமாக இருக்க வேண்டும்:

1. **காணக்கூடிய தன்மை** — முகவர் என்ன செய்கிறான், எவ்வளவு நேரம் எடுத்துக் கொள்கிறான், மற்றும் அது எந்த கருவிகளை அழைக்கிறது என்பதைக் காணக்கூடியதாக இருக்க வேண்டும். ட்ரேசிங் மற்றும் லாக்கிங் இல்லாமல், உற்பத்தி பிரச்சினைகளை பிழைதிருத்துவது சுமார் இயலாதது.

2. **மதிப்பீடு** — தானாக நடைபெறும் தரச் சோதனைகள் முகவரியின் பதில்கள் காலப்போக்கில் துல்லியமாகவும், முழுமையாகவும், உதவிகரமாகவும் இருக்குமென உறுதி செய்கின்றன. ஒரு மதிப்பாய்வு முகவர் நிர்ணயிக்கப்பட்ட அளவுகோல்களுக்கெதிராக பதில்களை மதிப்பெண்களால் அளிக்க முடியும்.

3. **செலவுக் கையாளுதல்** — டோக்கன் பயன்பாடு செலவுகளுக்கு நேரடியாக பாதிப்பை உருவாக்கிறது. ப்ராம்ட் மேம்படுத்தல், மாடல் தேர்வு மற்றும் கேஷிங் போன்ற வியூஹங்கள் செலவுகளை கட்டுப்பாட்டில் வைக்க உதவுகின்றன, தரத்தை யாரிடமும் இழக்காமல்.


## ஒரு கண்காணிக்கக்கூடிய முகவரை கட்டமைத்தல்

நாங்கள் பயண கருவிகளை வரையறுத்து, முகவர் அழைப்பை நேரம் கணக்கீட்டுடன் சுற்றிவைக்கிறோம், இதனால் தாமதத்தை கண்காணிக்க முடியும். உற்பத்தி சூழலில் நீங்கள் OpenTelemetry அல்லது இதே போன்ற ஒரு டிரேசிங் பேக்எண்டுடன் ஒருங்கிணைப்பீர்கள்.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## மதிப்பீட்டு மாதிரிகள்

பொதுவான உற்பத்தி நடைமுறையாக இரண்டாவது ஏஜென்டை ஒரு **மதிப்பீட்டாளராக** பயன்படுத்துவது உள்ளது. மதிப்பீட்டாளர் முதன்மை ஏஜென்டின் பதிலை முன்நிர்ணயிக்கப்பட்ட அளவுகோல்களான முழுமை, துல்லியம் மற்றும் உதவித்தன்மை போன்றவற்றிற்கு எதிராக மதிப்பிடுகிறார்.

இதன் மூலம்:
- பதில்கள் பயனர்களுக்கு செல்லும் முன் தானியங்கி தரக் கட்டுப்பாடுகள்
- ப்ராம்ப்டுகள் அல்லது மாடல்கள் மாற்றப்பட்டபோது செயல்திறன் குறைவு கண்டறிதல்
- காலப்பயணத்தின் போது ஏஜென்ட் செயல்திறனை தொடர்ச்சியாக கண்காணித்தல்


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## செலவு மேலாண்மை நெறிமுறைகள்

உற்பத்தி பயன்பாட்டில் உள்ள AI முகவரிகளுக்கான செலவுகளை கட்டுப்படுத்துவது மிகவும் முக்கியம். இங்கே முக்கியமான நெறிமுறைகள் உள்ளன:

| தந்திரம் | விளக்கம் |
|---|---|
| **ப்ராம்ட் திறன் மேம்பாடு** | சிஸ்டம் அறிவுறுத்தல்களை சுருக்கமாக வைத்திருங்கள். உள்ளீட்டு டோக்கன்களை குறைக்க தேவையற்ற பின்னணி தகவல்களை நீக்குங்கள். |
| **மாடல் தேர்வு** | வகைப்படுத்தல் அல்லது தகவல் எடுப்பு போன்ற எளிய பணிகளுக்கு (e.g. GPT-4o-mini) போன்ற சிறிய, மலிவான மாடல்களைப் பயன்படுத்தவும்; சிக்கலான விவேகத் திறனை talabசெய்து வேண்டிய கேள்விகளுக்கு பெரிய மாடல்களை ஒதுக்கி வைக்கவும். |
| **கேஷிங்** | கருவி முடிவுகளையும் அடிக்கடி கேட்கப்படும் கேள்விகளையும் கேஷ் செய்து, தேவையற்ற API அழைப்புகளைத் தவிர்க்குங்கள். |
| **டோக்கன் பட்ஜெட்டுகள்** | `max_tokens` வரம்புகளை அமைத்திடுங்கள், எதிர்பாராதமாக நீண்ட பதில்களை தவிர்க்க. |
| **குழு செயலாக்கம்** | சாத்தியமான இடங்களில் பல பயனர் கேள்விகளை ஒரு API அழைப்பாக குழுபடுத்துங்கள். |

In practice, a tiered approach works well: route straightforward requests to a fast, inexpensive model and escalate only complex queries to a more capable one.


## சுருக்கம்

இந்தப் பாடத்தில் நீங்கள் எப்படி செய்வதெல்லாம் என்பதை கற்றுக்கொண்டீர்கள்:

1. **கண்காணிப்புத்திறனை சேர்க்கவும்** — நேரம் மற்றும் பதிவுடன் ஏஜென்ட் தொடர்புகளில், தடையிடலும் மேற்பார்வையும் சாத்தியப்படுத்தும் அடித்தளத்தை அமைத்தல்.
2. **ஏஜென்ட் பதில்களை மதிப்பீடு செய்வது** — பூரணத்தன்மை, துல்லியம் மற்றும் உதவித்தன்மைக்கு மதிப்பெண் தரும் ஒரு மதிப்பீட்டாளர் ஏஜென்டைப் பயன்படுத்தி தானாக மதிப்பீடு செய்தல்.
3. **செலவுகளை நிர்வகிக்கவும்** — ப்ராம்ட் மேம்பாடு, மாதிரி தேர்வு, கேச்சிங் மற்றும் டோக்கன் பட்ஜெட்டுகள் மூலம் செலவுகளை கட்டுப்படுத்தல்.

இந்த தயாரிப்பு முறைகள் உங்கள் AI ஏஜென்டுகள் பரவலாக பயன்படுத்தப்படும் போது நம்பகமாகவும், அளவிடக்கூடியதாகவும், செலவுக்கு உகந்தவையாகவும் இருக்க உதவுகின்றன.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
பொறுப்பு விலக்கு:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவையாகிய Co‑op Translator (https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயல்வதற்குட்பட்டாலும், தானாக நடைபெறும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது துல்லியக்குறைவுகள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனத்தில் கொள்ளவும். அதன் தாய்மொழியில் உள்ள அசல் ஆவணம் அதிகாரபூர்வமான மூலமாக கருதப்பட வேண்டும். மிகவும் முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படக்கூடிய எந்த தவறான புரிதல்களுக்கும் அல்லது தவறான விளக்கங்களுக்கும் நாங்கள் பொறுப்பல்ல.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
